In [ ]:
import glob, os, re
import pandas as pd

d = os.getcwd()+"/"
PATH_RAW = f"{d}/../raw"

# ----------------------------------
# Append Google Trends

csv_files = glob.glob(os.path.join(PATH_RAW, "*multi*.csv"))
ts_files  = glob.glob(os.path.join(PATH_RAW, "time_series_*.csv"))

if len(csv_files) > 0:
    df0 = pd.DataFrame()
    for csv in csv_files:
        dfi = pd.read_csv(csv, skiprows=1)
        dfi.rename(columns={"Month": 'date'}, inplace=True)
        dfi.date = pd.to_datetime(dfi['date'])
        dfi.columns = dfi.columns.astype(str).str.lower().str.strip()
        dfi.columns = [re.sub(r'\W+', '_', col).strip('_') for col in dfi.columns]
        dfi.set_index('date', inplace=True)

        df0 = pd.concat([df0, dfi], axis=1)
    df0.columns = ["gtrends_" + col for col in df0.columns]

    # Replace zeroes with NaN
    df0.replace(0, pd.NA, inplace=True)

    df0.to_csv(f'{PATH_RAW}/gtrends.csv')
    print(f'Last updated: {df0.index.max()}')

    for file in csv_files:
        os.remove(file)

elif len(ts_files) > 0:
    df0 = pd.DataFrame()
    for csv in ts_files:
        # Extract country code from filename: time_series_TT_... -> "tt"
        basename = os.path.basename(csv)
        match = re.match(r'time_series_([A-Za-z]+)_', basename)
        country_suffix = f"_{match.group(1).lower()}" if match else ""

        dfi = pd.read_csv(csv, skiprows=0)
        dfi.rename(columns={"Time": 'date'}, inplace=True)
        dfi.date = pd.to_datetime(dfi['date'])
        dfi.columns = dfi.columns.astype(str).str.lower().str.strip()
        dfi.columns = [re.sub(r'\W+', '_', col).strip('_') for col in dfi.columns]
        dfi.set_index('date', inplace=True)

        # Append country code to each column
        if country_suffix:
            dfi.columns = [col + country_suffix for col in dfi.columns]

        df0 = pd.concat([df0, dfi], axis=1)
    df0.columns = ["gtrends_" + col for col in df0.columns]

    # Replace zeroes with NaN
    df0.replace(0, pd.NA, inplace=True)

    df0.to_csv(f'{PATH_RAW}/gtrends.csv')
    print(f'Last updated: {df0.index.max()}')

    for file in ts_files:
        os.remove(file)

df0.head(5)